# Experiment tracking on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/experiment-tracking.md`](../docs/notes/experiment-tracking.md)
for the verified SDK surface.

Experiment tracking has **two independent layers**:

- **Layer 1 — automatic.** Every tuning job streams train/validation curves to
  the Cloud console (**Agent Platform Studio → Tune and Distill → Monitor**) with
  no code. Nothing to do here.
- **Layer 2 — opt-in (this notebook).** Log your **own** params + offline metrics
  to **Vertex AI Experiments** to compare many runs, via the
  `geap_tuning.experiments` helper. Summary metrics are **free**; per-step
  time-series curves additionally need a **Managed TensorBoard** (cost +
  ~10–20 min provisioning), so TensorBoard is opt-in.

To keep cost down we reuse a **single** SFT-with-checkpoints job (mirroring
[`04_checkpoints.ipynb`](04_checkpoints.ipynb)), evaluate each checkpoint, and log
one Experiments run per checkpoint.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place. Keep the tuning/Experiments region aligned
> (`us-central1`).

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"  # supports intermediate checkpoints; tuning stays REGIONAL
EPOCHS = 3  # a few epochs -> a few checkpoints -> a few runs to compare
ADAPTER_SIZE = 8
DISPLAY_NAME = "geap-exp-tracking-sft"
EXPERIMENT_NAME = "geap-sft-checkpoint-eval"

cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build and stage the SFT dataset

Same deterministic support-intent splits as the SFT notebook.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.data import build_sft_dataset

paths = build_sft_dataset("../datasets/sft_support_intent")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/experiment_tracking_sft/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/experiment_tracking_sft/val.jsonl")
train_uri, val_uri

## 2. Tune once, keeping every checkpoint

Reuse an existing job with the same display name if one exists (cost control).

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.sft.tune import launch_sft_job

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        epochs=EPOCHS,
        adapter_size=ADAPTER_SIZE,
        export_last_checkpoint_only=False,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
job.state

## 3. Point Vertex AI Experiments at the tuning region

`init_experiment` creates/selects the experiment context so later `track_run`
calls attach to it. We pass no TensorBoard here — summary metrics don't need one.

In [ ]:
from geap_tuning.experiments import init_experiment

init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)

## 4. Evaluate every checkpoint on the held-out test split

Each checkpoint has its own endpoint, so we score them independently and collect
accuracy / macro-F1 per checkpoint.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import checkpoint_endpoint, list_checkpoints
from geap_tuning.sft.data import SUPPORT_TICKETS, build_records, split_dataset
from geap_tuning.sft.evaluate import run_eval

_, _, test_pairs = split_dataset(SUPPORT_TICKETS)
test_records = build_records(test_pairs)

results = []
for cp in list_checkpoints(job):
    endpoint = checkpoint_endpoint(job, cp.checkpoint_id)
    metrics = run_eval(
        test_records,
        predict_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
    )
    results.append((cp, metrics))
    print(f"checkpoint {cp.checkpoint_id} (epoch {cp.epoch}): acc={metrics['accuracy']:.3f}")

## 5. Log one summary run per checkpoint

`track_run` opens a run and logs its params; `log_summary_metrics` records one
value per key. No TensorBoard required — this alone gives a cross-run table.

In [ ]:
from geap_tuning.experiments import log_summary_metrics, track_run

for cp, metrics in results:
    params = {
        "base_model": BASE_MODEL,
        "epochs": EPOCHS,
        "adapter_size": ADAPTER_SIZE,
        "checkpoint_id": cp.checkpoint_id,
        "epoch": cp.epoch,
        "step": cp.step,
    }
    with track_run(f"{DISPLAY_NAME}-cp-{cp.checkpoint_id}", params=params):
        log_summary_metrics({"accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]})

## 6. (Optional) Managed TensorBoard time-series

> **Incurs cost and ~10–20 min provisioning.** Only run this cell if you want
> per-step curves in a user-owned TensorBoard.

Time-series metrics live in a Managed TensorBoard, so we provision/attach one
(reused by display name) and re-init the experiment with it before logging the
accuracy-vs-epoch curve as a single run.

In [ ]:
from geap_tuning.experiments import get_or_create_tensorboard, log_timeseries_metrics

tensorboard = get_or_create_tensorboard(
    "geap-tuning-tb", project=cfg.project, location=cfg.location, labels=cfg.labels
)
init_experiment(
    EXPERIMENT_NAME, project=cfg.project, location=cfg.location, tensorboard=tensorboard
)
with track_run(f"{DISPLAY_NAME}-curve"):
    for cp, metrics in sorted(results, key=lambda r: r[0].epoch):
        log_timeseries_metrics(
            {"accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]},
            step=cp.epoch,
        )
tensorboard

## 7. Compare runs

`experiment_dataframe` returns a pandas table of every run's params + summary
metrics — the same data you see under **Agent Platform Studio → Experiments**.

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)

## Next steps

Layer 1 (automatic console curves) and Layer 2 (this notebook) are
complementary. For the automatic metric keys per method and the full API surface
see [`docs/notes/experiment-tracking.md`](../docs/notes/experiment-tracking.md).